In [ ]:
import os
import duckdb
import ollama
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

con = duckdb.connect(str(PROJECT_ROOT / "data" / "f1.db"), read_only=True)
client = ollama.Client(host=os.environ.get("F1_OLLAMA_URL", "http://localhost:11434"))

print(f"DuckDB: {con.execute('SELECT COUNT(*) FROM f1_results').fetchone()[0]} rows")
print(f"Ollama: {[m.model for m in client.list().models]}")

In [ ]:
SCHEMA_CONTEXT = """You are a DuckDB SQL expert. Use ONLY the table and columns listed.

PRIMARY TABLE — use this for race results, wins, points, positions:
f1_results: season, round, race_name, driver_name, constructor, grid, position, points, status

ONLY use other tables when f1_results cannot answer the question:
f1_circuits: circuit_id, circuit_name, locality, country
f1_races: season, round, race_name, circuit_id, date
f1_qualifying: season, round, driver_id, position, q1, q2, q3
f1_driver_standings: season, position, points, wins, driver_id, constructor
f1_constructor_standings: season, position, points, wins, constructor_id, constructor

RULES:
- Always use f1_results for anything about wins, race positions, or points
- Filter wins with: TRY_CAST(position AS INTEGER) = 1
- Return ONLY raw SQL, nothing else

EXAMPLES:
Q: Who has the most race wins of all time?
A: SELECT driver_name, COUNT(*) AS wins FROM f1_results WHERE TRY_CAST(position AS INTEGER) = 1 GROUP BY driver_name ORDER BY wins DESC LIMIT 10

Q: Who won the 2021 Abu Dhabi Grand Prix?
A: SELECT driver_name, constructor FROM f1_results WHERE season = 2021 AND race_name LIKE '%Abu Dhabi%' AND TRY_CAST(position AS INTEGER) = 1

Q: Most wins for Ferrari in the 2000s?
A: SELECT driver_name, COUNT(*) AS wins FROM f1_results WHERE constructor = 'Ferrari' AND season BETWEEN 2000 AND 2009 AND TRY_CAST(position AS INTEGER) = 1 GROUP BY driver_name ORDER BY wins DESC"""

In [ ]:
def generate_sql(question: str) -> str:
    prompt = f"""{SCHEMA_CONTEXT}

Q: {question}
A:"""

    response = client.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    sql = response["message"]["content"].strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    # Take only first line if model adds explanation
    sql = sql.split("\n")[0].strip()
    
    return sql

In [ ]:
def execute_sql(sql: str) -> tuple:
    """
    Execute SQL against DuckDB. Returns (dataframe, error).
    """
    try:
        result = con.execute(sql).df()
        return result, None
    except Exception as e:
        return None, str(e)

# Test execution
result, error = execute_sql(sql)
if error:
    print(f"Error: {error}")
else:
    print(result)

In [ ]:
def sql_answer(question: str, result_df) -> str:
    """
    Given the question and SQL result, generate a natural language answer.
    """
    result_text = result_df.to_string(index=False)
    
    prompt = f"""You are an F1 expert. Answer the question based on this data.
Be concise and direct. Use the data to give a specific answer.

Question: {question}

Data:
{result_text}

Answer:"""

    response = client.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"].strip()

In [ ]:
def ask_sql(question: str, verbose: bool = True) -> str:
    """
    Full Text-to-SQL pipeline:
    1. Generate SQL from question
    2. Execute against DuckDB  
    3. Generate natural language answer
    """
    if verbose:
        print(f"Question: {question}")
    
    # Step 1: Generate SQL
    sql = generate_sql(question)
    if verbose:
        print(f"\nGenerated SQL:\n{sql}")
    
    # Step 2: Execute
    result, error = execute_sql(sql)
    if error:
        if verbose:
            print(f"\nSQL Error: {error}")
        return f"Could not execute query: {error}"
    
    if verbose:
        print(f"\nResult:\n{result}\n")
    
    # Step 3: Answer
    answer = sql_answer(question, result)
    if verbose:
        print(f"Answer:\n{answer}")
    
    return answer

In [ ]:
questions = [
    "Who has the most race wins of all time?",
    "Which constructor won the most races in 2022?",
    "Who won the 2021 Abu Dhabi Grand Prix?",  # This failed in Notebook 4
    "Who had the most wins for Ferrari in the 2000s?",  # This also failed
    "Which driver has the most podiums at Monaco?",
    "What is the average points scored by Mercedes per season?",
]

for q in questions:
    print("="*60)
    ask_sql(q)
    print()

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

# Load vector RAG components
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=str(PROJECT_ROOT / "chroma_db"))
collection = chroma_client.get_collection("f1_corpus")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def ask_vector(question: str) -> str:
    query_vec = embedding_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_vec, n_results=5)
    context = "\n".join([f"- {doc}" for doc in results["documents"][0]])
    
    prompt = f"""Answer using ONLY this context. Be concise.

Context:
{context}

Question: {question}
Answer:"""
    
    response = client.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"].strip()

# Compare both approaches on the same questions
comparison_questions = [
    "Who won the 2021 Abu Dhabi Grand Prix?",
    "Who had the most wins for Ferrari in the 2000s?",
    "Which constructor dominated the 2014 season?",
]

for q in comparison_questions:
    print("="*60)
    print(f"QUESTION: {q}")
    print(f"\nSQL PATH:    {ask_sql(q, verbose=False)}")
    print(f"\nVECTOR PATH: {ask_vector(q)}")
    print()

In [ ]:
# ## What We Learned

# ### SQL wins on:
# - Precise factual queries ("who won X race")
# - Aggregations ("most wins", "average points")
# - Cross-table joins ("Ferrari wins by driver")
# - Any question with a specific number in the answer

# ### Vector wins on:
# - Narrative questions ("how did X perform at Y")
# - Comparative/contextual questions
# - Questions where exact match matters less than semantic relevance

# ### The core insight:
# Text-to-SQL is more accurate but more brittle — one wrong column name 
# and the query fails. Vector RAG is more robust but less precise.

# The hybrid router in Notebook 6 combines both.